In [1]:
import requests
import pandas
from io import StringIO
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [2]:
url = "https://vtiav.sm.ee/index.php/opendata/supluskohad.xml"


In [3]:
data = requests.get(url).text

In [4]:
df = pandas.read_xml(
    StringIO(data),
    xpath=".//supluskoht"  # assumes each bathing site is under <supluskoht>
)

coords = pandas.read_xml(
    StringIO(data),
    xpath=".//supluskoht/koordinaadid/koordinaat"
)

# If multiple coordinates per site exist, take the first
coords = coords.groupby(coords.index).first().reset_index(drop=True)

# Combine and clean
result = pandas.DataFrame({
    "id": df["id"],
    "name": df["nimetus"],
    "x": coords["x"],
    "y": coords["y"],
    "address": df["aadress"]
})
result

,id,name,x,y,address
0,328,Aablahe rand,6605088.70,586304.150,"Mardi, 74717 Kolga-Aabla küla, Kuusalu vald, H..."
1,124,Aafrika rand,6533928.61,472585.550,"Haapsalu linn, Haapsalu linn"
2,461,Aa rand,6513518.06,630068.432,"Aa küla, Lüganuse vald, Ida-Viru maakond"
3,464,Aidu karjääri supluskoht,6545887.55,696748.430,"Paadi, 42305 Aidu küla, Lüganuse vald, Ida-Vir..."
4,86,Aidu tehisjärve supluskoht,6568002.82,575962.930,"Aidu küla, Põltsamaa vald, Jõgeva maakond"
...,...,...,...,...,...
208,376,Viitna Pikkjärv,NaN,NaN,"Loobu metskond 71, 45202 Viitna küla, Kadrina ..."
209,74,Viljandi järve supluskoht,NaN,NaN,Viljandi linn
210,499,Võnnu paisjärv,NaN,NaN,"Võnnu alevik, Kastre vald, Tartu maakond"
211,340,"Võrtsjärve, Trepimäe supluskoht",NaN,NaN,"Trepimäe puhkeala, 61117 Vehendi küla, Elva va..."


In [5]:
locator = Nominatim(user_agent='zwemwater', timeout=10)
geocode = RateLimiter(locator.geocode, min_delay_seconds=1)

In [6]:
from tqdm.notebook import tqdm

tqdm.pandas()
geocodeDF = result[result['x'].isna()]
geocodeDF['location'] = geocodeDF['address'].progress_apply(geocode)
geocodeDF

  0%|          | 0/51 [00:00<?, ?it/s]

,id,name,x,y,address,location
162,232,Roosna-Alliku tehisjärv,NaN,NaN,"Järve, 73203 Allikjärve küla, Paide linn","(Järve, Kaltenbrunni matkarada, Allikjärve kül..."
163,155,Roosta rand,NaN,NaN,"Elbiku küla / Ölbäck, Lääne-Nigula vald, Lääne...","(Elbiku küla / Ölbäck, Lääne-Nigula vald, Lään..."
164,249,Ropka järv,NaN,NaN,"Külitse alevik, Kambja vald, Tartu maakond","(Külitse alevik, Kambja vald, Tartu maakond, 6..."
165,178,Rõuge Suurjärve supluskoht,NaN,NaN,"Suurjärv, Rõuge alevik, Rõuge vald, Võru maakond","(Suurjärv, Rõuge alevik, Rõuge, Rõuge vald, Võ..."
166,82,Saadjärve Kukulinna,NaN,NaN,"Kukulinna küla, Tartu vald, Tartu maakond","(Kukulinna küla, Tartu vald, Tartu maakond, 60..."
167,85,Saadjärve Tabivere supluskoht,NaN,NaN,"Tabivere alevik, Tartu vald, Tartu maakond","(Tabivere alevik, Tartu vald, Tartu maakond, 4..."
168,226,Salmistu ranna parempoolne ala,NaN,NaN,"Salmistu küla, Kuusalu vald, Harju maakond","(Salmistu küla, Kuusalu vald, Harju maakond, 7..."
169,128,Sillamäe supluskoht,NaN,NaN,Sillamäe linn,"(Sillamäe linn, Ida-Viru maakond, Eesti, (59.3..."
170,148,Sindi väliujula (Pärnu jõgi),NaN,NaN,"Sindi linn, Tori vald, Pärnu maakond","(Sindi linn, Tori vald, Pärnu maakond, Eesti, ..."
171,210,Suure-Jaani paisjärv,NaN,NaN,"Suure-Jaani linn, Põhja-Sakala vald, Viljandi ...","(Suure-Jaani linn, Põhja-Sakala vald, Viljandi..."


In [7]:
geocodeDF['x'] = geocodeDF['location'].apply(lambda loc: loc.latitude if loc else None)
geocodeDF['y'] = geocodeDF['location'].apply(lambda loc: loc.longitude if loc else None)
geocodeDF

,id,name,x,y,address,location
162,232,Roosna-Alliku tehisjärv,59.020029,25.698037,"Järve, 73203 Allikjärve küla, Paide linn","(Järve, Kaltenbrunni matkarada, Allikjärve kül..."
163,155,Roosta rand,59.155928,23.549833,"Elbiku küla / Ölbäck, Lääne-Nigula vald, Lääne...","(Elbiku küla / Ölbäck, Lääne-Nigula vald, Lään..."
164,249,Ropka järv,58.317802,26.611083,"Külitse alevik, Kambja vald, Tartu maakond","(Külitse alevik, Kambja vald, Tartu maakond, 6..."
165,178,Rõuge Suurjärve supluskoht,57.727904,26.921792,"Suurjärv, Rõuge alevik, Rõuge vald, Võru maakond","(Suurjärv, Rõuge alevik, Rõuge, Rõuge vald, Võ..."
166,82,Saadjärve Kukulinna,58.516296,26.715720,"Kukulinna küla, Tartu vald, Tartu maakond","(Kukulinna küla, Tartu vald, Tartu maakond, 60..."
167,85,Saadjärve Tabivere supluskoht,58.555453,26.588104,"Tabivere alevik, Tartu vald, Tartu maakond","(Tabivere alevik, Tartu vald, Tartu maakond, 4..."
168,226,Salmistu ranna parempoolne ala,59.486353,25.380737,"Salmistu küla, Kuusalu vald, Harju maakond","(Salmistu küla, Kuusalu vald, Harju maakond, 7..."
169,128,Sillamäe supluskoht,59.396265,27.762556,Sillamäe linn,"(Sillamäe linn, Ida-Viru maakond, Eesti, (59.3..."
170,148,Sindi väliujula (Pärnu jõgi),58.407513,24.658316,"Sindi linn, Tori vald, Pärnu maakond","(Sindi linn, Tori vald, Pärnu maakond, Eesti, ..."
171,210,Suure-Jaani paisjärv,58.537376,25.464948,"Suure-Jaani linn, Põhja-Sakala vald, Viljandi ...","(Suure-Jaani linn, Põhja-Sakala vald, Viljandi..."
